[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CkCreative/transformers-and-math-notebooks/blob/main/21-preference-learning.ipynb)

# Module 21: Preference Learning: RL, RLHF and DPO

Companion notebook to [Module 21](https://transformers-and-math.drmhse.com/modules/21-preference-learning) of *Transformers & Math*. Run the cells in order; each part names the section of the module it goes with. Everything runs on a free Colab CPU in under a minute or two.

In [ ]:
import math
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

## Part 1 — REINFORCE, with and without a baseline (section 21.2)
A three-armed bandit with average rewards 5.0, 5.5 and 6.0.

In [ ]:
means = torch.tensor([5.0, 5.5, 6.0])

def reinforce(baseline, seed, steps=400, lr=0.02):
    g = torch.Generator().manual_seed(seed)
    logits, avg, trace = torch.zeros(3), 0.0, []
    for _ in range(steps):
        p = torch.softmax(logits, 0)
        a = torch.multinomial(p, 1, generator=g).item()
        r = means[a] + torch.randn(1, generator=g).item()
        adv = r - avg if baseline else r
        avg += 0.05 * (r - avg)
        logits += lr * adv * (F.one_hot(torch.tensor(a), 3) - p)      # ∇ log π(a) = one-hot − p
        trace.append(p[2].item())
    return trace

for baseline, style in ((False, 'r-'), (True, 'g-')):
    finals = torch.tensor([reinforce(baseline, seed)[-1] for seed in range(10)])
    for seed in range(4):
        plt.plot(reinforce(baseline, seed), style, alpha=0.5)
    print(f'baseline={baseline!s:5}: P(best) after 400 steps over 10 seeds: mean {finals.mean():.2f}, spread (sd) {finals.std(unbiased=False):.2f}')
plt.ylabel('P(best action)'); plt.title('red: no baseline   green: baseline'); plt.show()

## Part 2 — PPO's clipped objective (section 21.3)

In [ ]:
eps = 0.2
for rho in (0.7, 1.0, 1.15, 1.5):
    for A in (1.0, -1.0):
        obj = min(rho * A, max(min(rho, 1 + eps), 1 - eps) * A)
        print(f'ratio {rho:4}, advantage {A:+}: objective {obj:+.2f}')

## Part 3 — A reward model from preferences (section 21.4)
Fit Bradley–Terry rewards for five responses from simulated human comparisons.

In [ ]:
torch.manual_seed(0)
true_r = torch.tensor([0.0, 1.0, 2.0, 0.5, -1.0])
a, b = torch.randint(0, 5, (4000,)), torch.randint(0, 5, (4000,))
keep = a != b; a, b = a[keep], b[keep]
a_wins = torch.rand(len(a)) < torch.sigmoid(true_r[a] - true_r[b])
w, l = torch.where(a_wins, a, b), torch.where(a_wins, b, a)
r = torch.zeros(5, requires_grad=True)
opt = torch.optim.Adam([r], lr=0.05)
for _ in range(500):
    loss = -F.logsigmoid(r[w] - r[l]).mean()
    opt.zero_grad(); loss.backward(); opt.step()
print('fitted rewards (shifted):', [round(v, 2) for v in (r - r[0]).tolist()], ' true:', true_r.tolist())

## Part 4 — DPO recovers the optimal policy (section 21.6)
Train a policy over the five responses on the same preference pairs with the DPO loss. Theory says it ends at $\pi^* \propto \pi_{\text{ref}} \exp(r/\beta)$.

In [ ]:
beta = 0.5
ref = torch.tensor([0.3, 0.25, 0.1, 0.2, 0.15])
logits = ref.log().clone().requires_grad_(True)
r = (r - r[0]).detach()
opt = torch.optim.Adam([logits], lr=0.05)
for _ in range(1000):
    logp = F.log_softmax(logits, 0)
    margin = beta * ((logp[w] - ref.log()[w]) - (logp[l] - ref.log()[l]))
    loss = -F.logsigmoid(margin).mean()
    opt.zero_grad(); loss.backward(); opt.step()
def optimal(rewards):
    t = ref * torch.exp(rewards / beta)
    return [round(v, 3) for v in (t / t.sum()).tolist()]
print('DPO-trained policy:                 ', [round(v, 3) for v in torch.softmax(logits, 0).tolist()])
print('π* from the rewards fitted in Part 3:', optimal(r.detach()))
print('π* from the true rewards:           ', optimal(true_r))
# The first two agree: DPO finds the policy that is optimal for the Bradley–Terry rewards its data implies.
# Both differ a little from the true optimum because 4,000 comparisons pin the rewards down only approximately.

## Your turn
1. Raise β to 2. How does the DPO policy change, and why does β act like the strength of the KL penalty?
2. In Part 1, add 100 to every reward. What happens to the runs without a baseline? With one?